In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

# Set configurations
vocab_size = 10000  # Top 10,000 frequent words
max_length = 200    # Cut off reviews after 200 words

# Load training and testing data
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)
print(f"Train sequences: {len(X_train)}, Test sequences: {len(X_test)}")


Train sequences: 25000, Test sequences: 25000


In [7]:
# Pad sequences to a fixed length
X_train = pad_sequences(X_train, maxlen=max_length, padding='post')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post')


In [8]:
# Initialize sequential framework
model = Sequential([
   Input(shape = (max_length,)),
    # Layer 1: Word Embedding (turns word indices into 32-dimensional dense vectors)
    Embedding(input_dim=vocab_size, output_dim=32),

    # Layer 2: Recurrent Neural Network layer with 64 memory units
    SimpleRNN(64, activation='tanh', return_sequences=False),

    # Layer 3: Dropout to minimize overfitting
    Dropout(0.5),

    # Layer 4: Output Layer for binary classification
    Dense(1, activation='sigmoid')
])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 32)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 326,273 (1.24 MB)

 Trainable params: 326,273 (1.24 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Compile the network
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the network
history = model.fit(X_train, y_train,
                    epochs=5,
                    batch_size=64,
                    validation_split=0.2)


Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 0.5087 - loss: 0.7007 - val_accuracy: 0.5224 - val_loss: 0.6895
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.5644 - loss: 0.6750 - val_accuracy: 0.5472 - val_loss: 0.6760
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.6206 - loss: 0.6448 - val_accuracy: 0.5376 - val_loss: 0.6795
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.6610 - loss: 0.5763 - val_accuracy: 0.5532 - val_loss: 0.6844
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.7151 - loss: 0.4790 - val_accuracy: 0.5440 - val_loss: 0.7483


In [10]:
# Evaluate model on test split
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")


782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.5416 - loss: 0.7538
Test Accuracy: 54.16%
